
# 04 - AcmeNet RAG-style Chatbot Demo

Goal:
Build a simple retrieval-grounded chatbot workflow using AcmeNet support chunks stored in Databricks.

Input:
acmenet_silver_chunks

Output:
acmenet_gold_chatbot_interactions

In [0]:
silver_df = spark.table("acmenet_silver_chunks")

display(
   silver_df.select(
        "chunk_id",
        "document_name",
        "section",
        "source",
        "chunk_text"
    ) 
)

chunk_id,document_name,section,source,chunk_text
troubleshooting_guide_001,troubleshooting_guide.md,Slow Internet,troubleshooting,"Customers experiencing slow internet should restart the modem by unplugging it for 30 seconds. If speed remains below 50% of the contracted plan after restart, escalate to Tier 2 support."
billing_policy_001,billing_policy.md,Billing Disputes,billing,Customers can request a billing clarification within 15 days of receiving the invoice. Billing disputes above 100 dollars must be escalated to Tier 2 support.
refund_policy_001,refund_policy.md,Refund Eligibility,refunds,Customers may request a refund within 30 days of activation if the service was unavailable for more than 72 continuous hours due to a provider-side issue.


In [0]:
# Validate Knwoledge base
chunk_count = silver_df.count()

print(f"Knowledge chunks available: {chunk_count}")

if chunk_count == 0:
    raise ValueError("No knowledge chunks found. Please run Notebook 02 first.")

Knowledge chunks available: 3


In [0]:
# Extract keywords function

import re
import json
from datetime import datetime

STOPWORDS = {
    "the", "is", "are", "a", "an", "and", "or", "to", "of", "in",
    "my", "i", "what", "should", "do", "can", "if", "it", "for",
    "with", "on", "this", "that", "be", "you", "your", "me"
}

def extract_keywords(question: str) -> list[str]:
    normalized = question.lower()
    normalized = re.sub(r"[^a-z0-9\s]", " ", normalized)
    words = normalized.split()

    keywords = [
        word for word in words
        if word not in STOPWORDS and len(word) >= 3
    ]

    return keywords

In [0]:
# Retrieval function
def retrieve_chunks(question: str, top_k: int = 3) -> list[dict]:
    keywords = extract_keywords(question)

    if not keywords:
        return []

    rows = silver_df.select(
        "chunk_id",
        "document_name",
        "section",
        "source",
        "chunk_text"
    ).collect()

    scored_chunks = []

    for row in rows:
        chunk_text = row["chunk_text"]
        chunk_text_lower = chunk_text.lower()

        matched_keywords = [
            keyword for keyword in keywords
            if keyword in chunk_text_lower
        ]

        score = len(matched_keywords)

        if score > 0:
            scored_chunks.append({
                "chunk_id": row["chunk_id"],
                "document_name": row["document_name"],
                "section": row["section"],
                "source": row["source"],
                "chunk_text": chunk_text,
                "matched_keywords": matched_keywords,
                "score": score
            })

    scored_chunks = sorted(
        scored_chunks,
        key=lambda item: item["score"],
        reverse=True
    )

    return scored_chunks[:top_k]

In [0]:
# Build context function
def build_context(retrieved_chunks: list[dict]) -> str:
    context_blocks = []

    for chunk in retrieved_chunks:
        context_blocks.append(
            f"Source: {chunk['document_name']}\n"
            f"Section: {chunk['section']}\n"
            f"Content: {chunk['chunk_text']}"
        )

    return "\n\n---\n\n".join(context_blocks)

In [0]:
# Answer question function
def answer_question(question: str) -> dict:
    retrieved_chunks = retrieve_chunks(question)

    if not retrieved_chunks:
        return {
            "question": question,
            "answer": (
                "I can only answer questions related to AcmeNet support policies, "
                "troubleshooting, billing, refunds, installation, and escalation rules."
            ),
            "sources": [],
            "status": "out_of_domain",
            "retrieved_context": ""
        }

    context = build_context(retrieved_chunks)

    answer = (
        "Based on AcmeNet support documentation, here is the most relevant information:\n\n"
        + "\n\n".join([chunk["chunk_text"] for chunk in retrieved_chunks])
    )

    sources = sorted(
        list(set([chunk["document_name"] for chunk in retrieved_chunks]))
    )

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "status": "answered",
        "retrieved_context": context
    }

In [0]:
# Test quesiton
result = answer_question("My internet is slow. What should I do?")

print("Question:")
print(result["question"])

print("\nAnswer:")
print(result["answer"])

print("\nSources:")
print(result["sources"])

print("\nStatus:")
print(result["status"])

Question:
My internet is slow. What should I do?

Answer:
Based on AcmeNet support documentation, here is the most relevant information:

Customers experiencing slow internet should restart the modem by unplugging it for 30 seconds. If speed remains below 50% of the contracted plan after restart, escalate to Tier 2 support.

Sources:
['troubleshooting_guide.md']

Status:
answered


In [0]:
# See retrieved context
print(result["retrieved_context"])

Source: troubleshooting_guide.md
Section: Slow Internet
Content: Customers experiencing slow internet should restart the modem by unplugging it for 30 seconds. If speed remains below 50% of the contracted plan after restart, escalate to Tier 2 support.


In [0]:
# Save interaction in Golden Table

interaction_record = [{
    "question": result["question"],
    "answer": result["answer"],
    "sources_json": json.dumps(result["sources"]),
    "status": result["status"],
    "retrieved_context": result["retrieved_context"],
    "created_at_utc": datetime.utcnow().isoformat()
}]

interaction_df = spark.createDataFrame(interaction_record)

interaction_df.write.format("delta").mode("append").saveAsTable(
    "acmenet_gold_chatbot_interactions"
)

print("Interaction saved to acmenet_gold_chatbot_interactions")

/home/spark-37bddb94-bdf0-482c-81c4-b9/.ipykernel/2080/command-7785304000477324-1686838816:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at_utc": datetime.utcnow().isoformat()


Interaction saved to acmenet_gold_chatbot_interactions


In [0]:
# Read interactions history
gold_df = spark.table("acmenet_gold_chatbot_interactions")

display(
    gold_df.select(
        "question",
        "answer",
        "sources_json",
        "status",
        "created_at_utc"
    )
)

question,answer,sources_json,status,created_at_utc
My internet is slow. What should I do?,"Based on AcmeNet support documentation, here is the most relevant information: Customers experiencing slow internet should restart the modem by unplugging it for 30 seconds. If speed remains below 50% of the contracted plan after restart, escalate to Tier 2 support.","[""troubleshooting_guide.md""]",answered,2026-05-23T08:11:14.657674
My internet is slow. What should I do?,"Based on AcmeNet support documentation, here is the most relevant information: Customers experiencing slow internet should restart the modem by unplugging it for 30 seconds. If speed remains below 50% of the contracted plan after restart, escalate to Tier 2 support.","[""troubleshooting_guide.md""]",answered,2026-05-23T08:10:05.637747


In [0]:
# Create a function for ask and save
def ask_and_log(question: str) -> dict:
    result = answer_question(question)

    interaction_record = [{
        "question": result["question"],
        "answer": result["answer"],
        "sources_json": json.dumps(result["sources"]),
        "status": result["status"],
        "retrieved_context": result["retrieved_context"],
        "created_at_utc": datetime.utcnow().isoformat()
    }]

    interaction_df = spark.createDataFrame(interaction_record)

    interaction_df.write.format("delta").mode("append").saveAsTable(
        "acmenet_gold_chatbot_interactions"
    )


    return result

In [0]:
# Test asking multiple questions

questions = [
    "My internet is slow. What should I do?",
    "Can I dispute a charge on my invoice?",
    "Can I get a refund if my service was down for more than 72 hours?",
    "Can you recommend a gaming laptop?"
]

for question in questions:
    result = ask_and_log(question)

    print("=" * 80)
    print("Question:", result["question"])
    print("Status:", result["status"])
    print("Sources:", result["sources"])
    print("Answer:", result["answer"])


/home/spark-37bddb94-bdf0-482c-81c4-b9/.ipykernel/2080/command-7785304000477326-3657867894:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at_utc": datetime.utcnow().isoformat()


Question: My internet is slow. What should I do?
Status: answered
Sources: ['troubleshooting_guide.md']
Answer: Based on AcmeNet support documentation, here is the most relevant information:

Customers experiencing slow internet should restart the modem by unplugging it for 30 seconds. If speed remains below 50% of the contracted plan after restart, escalate to Tier 2 support.
Question: Can I dispute a charge on my invoice?
Status: answered
Sources: ['billing_policy.md']
Answer: Based on AcmeNet support documentation, here is the most relevant information:

Customers can request a billing clarification within 15 days of receiving the invoice. Billing disputes above 100 dollars must be escalated to Tier 2 support.
Question: Can I get a refund if my service was down for more than 72 hours?
Status: answered
Sources: ['refund_policy.md']
Answer: Based on AcmeNet support documentation, here is the most relevant information:

Customers may request a refund within 30 days of activation if the

In [0]:
%sql

SELECT
  status,
  COUNT(*) AS total_interactions
FROM acmenet_gold_chatbot_interactions
GROUP BY status;

status,total_interactions
answered,5
out_of_domain,1


In [0]:
%sql

SELECT
  question,
  sources_json,
  status,
  created_at_utc
FROM acmenet_gold_chatbot_interactions
ORDER BY created_at_utc DESC;

question,sources_json,status,created_at_utc
Can you recommend a gaming laptop?,[],out_of_domain,2026-05-23T08:13:46.734809
Can I get a refund if my service was down for more than 72 hours?,"[""refund_policy.md""]",answered,2026-05-23T08:13:44.248553
Can I dispute a charge on my invoice?,"[""billing_policy.md""]",answered,2026-05-23T08:13:41.906661
My internet is slow. What should I do?,"[""troubleshooting_guide.md""]",answered,2026-05-23T08:13:39.496899
My internet is slow. What should I do?,"[""troubleshooting_guide.md""]",answered,2026-05-23T08:11:14.657674
My internet is slow. What should I do?,"[""troubleshooting_guide.md""]",answered,2026-05-23T08:10:05.637747


In [0]:
%sql

SELECT
  question,
  answer,
  created_at_utc
FROM acmenet_gold_chatbot_interactions
WHERE status = 'out_of_domain'
ORDER BY created_at_utc DESC;

question,answer,created_at_utc
Can you recommend a gaming laptop?,"I can only answer questions related to AcmeNet support policies, troubleshooting, billing, refunds, installation, and escalation rules.",2026-05-23T08:13:46.734809
